In [4]:
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-1b-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for use! \n\nI’m designed to take text and images as input and generate text as output. \n\nHow can I help you today?'}]}]

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for anyone to use! 

I can take text and images as input and provide text-based responses. 

What’s up? 😊<end_of_turn>


In [6]:
%%writefile scorer.py

import re
import sys
import unicodedata

def normalize(s: str) -> str:
    """Lowercase, strip unicode quirks, remove trailing punctuation, collapse whitespace."""
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = s.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    s = re.sub(r"^(answer|the answer is)\s*[:\-]?\s*", "", s)
    s = re.sub(r"[.,;:!?]+$", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_number(s: str):
    """Canonicalize a numeric string to 3-decimal form. Returns None if not numeric.
    Accepts '0.312', '.312', '0.3120', '31.2%' -> all become '0.312'.
    """
    s = s.strip()
    pct = s.endswith("%")
    if pct:
        s = s[:-1].strip()
    m = re.fullmatch(r"(-?\d*\.?\d+)", s)
    if not m:
        return None
    try:
        val = float(m.group(1))
    except ValueError:
        return None
    if pct:
        val /= 100.0
    return f" {val:.3f}".strip()

def score_mc(pred: str, gold: str) -> bool:
    """Gold is a single letter A/B/C/D. Take the first standalone A/B/C/D in the prediction."""
    m = re.search(r"\b([ABCD])\b", pred.upper())
    return bool(m) and m.group(1) == gold.strip().upper()

def score_short(pred: str, gold: str) -> bool:
    """Exact match after normalization; numeric values compared to 3 decimals."""
    p, g = normalize(pred), normalize(gold)
    if p == g:
        return True
    pn, gn = normalize_number(p), normalize_number(g)
    if pn is not None and gn is not None:
        return pn == gn
    return False

def score(pred: str, gold: str, qtype: str) -> bool:
    if qtype == "mc":
        return score_mc(pred, gold)
    # "short" and "stats" both go through score_short
    return score_short(pred, gold)

def _run_tests():
    cases_short = [
        ("0.312", "0.312", True),
        (".312", "0.312", True),
        ("31.2%", "0.312", True),
        ("0.3120", "0.312", True),
        ("The answer is 0.312.", "0.312", True),
        ("0.31", "0.312", False),
        ("Babe Ruth", "babe ruth", True),
        ("Ruth", "Babe Ruth", False),
        ("  Babe   Ruth  ", "Babe Ruth", True),
    ]
    cases_mc = [
        ("A", "A", True),
        ("a.", "A", True),
        ("The answer is C", "C", True),
        ("B because ...", "B", True),
        ("D", "A", False),
        ("I don't know", "A", False),
    ]

    failures = 0
    for pred, gold, expected in cases_short:
        got = score_short(pred, gold)
        if got != expected:
            failures += 1
            print(f"FAIL short: pred={pred!r} gold={gold!r} expected={expected} got={got}")
    for pred, gold, expected in cases_mc:
        got = score_mc(pred, gold)
        if got != expected:
            failures += 1
            print(f"FAIL mc:    pred={pred!r} gold={gold!r} expected={expected} got={got}")

    if failures == 0:
        print(f"All {len(cases_short) + len(cases_mc)} scorer tests passed.")
    else:
        print(f"{failures} test(s) FAILED.")
    return failures == 0

def test_short():
    for pred, gold, expected in [
        ("0.312", "0.312", True), (".312", "0.312", True),
        ("31.2%", "0.312", True), ("0.3120", "0.312", True),
        ("The answer is 0.312.", "0.312", True), ("0.31", "0.312", False),
        ("Babe Ruth", "babe ruth", True), ("Ruth", "Babe Ruth", False),
        ("  Babe   Ruth  ", "Babe Ruth", True),
    ]:
        assert score_short(pred, gold) == expected, (pred, gold, expected)


def test_mc():
    for pred, gold, expected in [
        ("A", "A", True), ("a.", "A", True),
        ("The answer is C", "C", True), ("B because ...", "B", True),
        ("D", "A", False), ("I don't know", "A", False),
    ]:
        assert score_mc(pred, gold) == expected, (pred, gold, expected)


if __name__ == "__main__":
    ok = _run_tests()
    if 'ipykernel' not in sys.modules:
        sys.exit(0 if ok else 1)

Writing scorer.py


In [9]:

import os, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

from scorer import score

MODEL_ID = "google/gemma-3-1b-it"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

KNOWLEDGE_ITEMS = [
    # (question, gold, qtype)
    ("How many players are on the field for one team in baseball at the start of a play?\nA. 8\nB. 9\nC. 10\nD. 11", "B", "mc"),
    ("How many strikes make a strikeout?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
    ("How many balls make a walk?\nA. 3\nB. 4\nC. 5\nD. 6", "B", "mc"),
    ("Which team won the 2016 World Series?\nA. Cleveland Indians\nB. Chicago Cubs\nC. Boston Red Sox\nD. Los Angeles Dodgers", "B", "mc"),
    ("Who holds the MLB career home run record?", "Barry Bonds", "short"),
    ("How many innings are in a standard MLB game?\nA. 7\nB. 8\nC. 9\nD. 10", "C", "mc"),
    ("What is it called when a batter hits and reaches all four bases in one play?\nA. Triple\nB. Home run\nC. Grand slam\nD. Cycle", "B", "mc"),
    ("Which league uses the designated hitter in all games as of 2022?\nA. National League\nB. American League\nC. Both\nD. Neither", "C", "mc"),
    ("Who is the all-time MLB hits leader?", "Pete Rose", "short"),
    ("How many outs end a half-inning?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
]

# 10 stats questions. Counts are embedded -> pure calculation, per the plan.
STATS_ITEMS = [
    ("Using these career totals: AB=1000, H=300, BB=100, HBP=5, SF=5. What is the career OBP? Round to 3 decimals.", "0.370", "stats"),
    ("Using these career totals: AB=800, H=200, BB=80, HBP=0, SF=0. What is the career OBP? Round to 3 decimals.", "0.318", "stats"),
    ("Using these career totals: AB=500, H=150, 2B=30, 3B=5, HR=20. What is the career SLG? Round to 3 decimals.", "0.500", "stats"),
    ("Using these career totals: AB=400, H=100, 2B=20, 3B=2, HR=10. What is the career SLG? Round to 3 decimals.", "0.385", "stats"),
    ("Using these career totals: AB=600, H=180. What is the career AVG? Round to 3 decimals.", "0.300", "stats"),
    ("Using these career totals: AB=900, H=270, BB=90, HBP=5, SF=5. What is the career OBP? Round to 3 decimals.", "0.370", "stats"),
    ("Using these career totals: AB=700, H=210, 2B=40, 3B=4, HR=30. What is the career SLG? Round to 3 decimals.", "0.497", "stats"),
    ("Using these career totals: AB=300, H=90. What is the career AVG? Round to 3 decimals.", "0.300", "stats"),
    ("Using these career totals: AB=1000, H=250, BB=150, HBP=10, SF=10. What is the career OBP? Round to 3 decimals.", "0.350", "stats"),
    ("Using these career totals: AB=550, H=140, 2B=25, 3B=3, HR=15. What is the career SLG? Round to 3 decimals.", "0.393", "stats"),
]

print("Loading tokenizer + model (fp16 on T4)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=userdata.get("HF_TOKEN"),
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Loaded. GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def generate(prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def run(items, label):
    correct, results = 0, []
    t0 = time.time()
    for q, gold, qtype in items:
        pred = generate(q)
        ok = score(pred, gold, qtype)
        correct += ok
        results.append((q[:60], gold, pred[:60], ok))
    dt = time.time() - t0
    acc = correct / len(items)
    print(f"\n=== {label}: {correct}/{len(items)} = {acc:.0%}  ({dt:.1f}s) ===")
    for q, gold, pred, ok in results:
        mark = "✓" if ok else "✗"
        print(f"  {mark} gold={gold!r:>14}  pred={pred!r}")
    return acc

print("\n" + "="*60)
acc_know = run(KNOWLEDGE_ITEMS, "KNOWLEDGE (base, zero-shot)")
acc_stats = run(STATS_ITEMS,    "STATS (base, zero-shot, counts in prompt)")

print("\n" + "="*60)
print(f"Knowledge accuracy: {acc_know:.0%}")
print(f"Stats accuracy:     {acc_stats:.0%}")

if acc_know < 0.20:
    print("\nVERDICT: knowledge is near the floor (<20%).")
    print("  -> A 1B model likely lacks the latent knowledge to 'unlock'.")
    print("  -> SWITCH to a structured classification/extraction task type.")
elif acc_know > 0.85:
    print("\nVERDICT: knowledge is too easy (>85%).")
    print("  -> Base already strong; LoRA delta will be hard to see.")
    print("  -> ADD harder items or change task type.")
else:
    print("\nVERDICT: knowledge is in the 20-85% band — a usable 'middle'.")
    print("  -> KEEP baseball; cut stats to ~10-15% of test, report separately.")
    print("  -> Proceed to dataset build.")
print(f"\nGPU peak so far: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Loading tokenizer + model (fp16 on T4)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded. GPU mem after load: 6.06 GB


=== KNOWLEDGE (base, zero-shot): 6/10 = 60%  (25.6s) ===
  ✓ gold=           'B'  pred="The correct answer is **B. 9**\n\nHere's why:\n\n*   **Starting "
  ✓ gold=           'B'  pred='The correct answer is **B. 3**\n\nA strikeout is when a pitche'
  ✓ gold=           'B'  pred='The correct answer is **B. 4**\n\nA walk is typically made up '
  ✓ gold=           'B'  pred='The correct answer is **B. Chicago Cubs**.\n\nThe Chicago Cubs'
  ✗ gold= 'Barry Bonds'  pred='As of today, November 2, 2023, the MLB career home run recor'
  ✗ gold=           'C'  pred="The correct answer is **B. 8**\n\nHere's how innings are calcu"
  ✗ gold=           'B'  pred="The correct answer is **D. Cycle**.\n\nHere's why:\n\n*   **Cycl"
  ✓ gold=           'C'  pred='The correct answer is **C. Both**\n\nAs of 2022, the MLB utili'
  ✗ gold=   'Pete Rose'  pred='As of today, November 2, 2023, the all-time MLB hits leader '
  ✓ gold=           'B'  pred="The correct answer 

In [12]:
# probe_base_model_v2.py — revised 10-item probe
# bare-answer prompting for short items; formulas defined

import os, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scorer import score

MODEL_ID = "google/gemma-3-1b-it"
HF_TOKEN = os.environ.get("HF_TOKEN") or "PASTE_YOUR_TOKEN_HERE"

BARE = " Answer with just the name or phrase, no explanation."
NUM  = " Give only the number, rounded to 3 decimals, no explanation."

KNOWLEDGE_ITEMS = [
    ("How many players are on the field for one team in baseball at the start of a play?\nA. 8\nB. 9\nC. 10\nD. 11", "B", "mc"),
    ("How many strikes make a strikeout?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
    ("How many balls make a walk?\nA. 3\nB. 4\nC. 5\nD. 6", "B", "mc"),
    ("Which team won the 2016 World Series?\nA. Cleveland Indians\nB. Chicago Cubs\nC. Boston Red Sox\nD. Los Angeles Dodgers", "B", "mc"),
    ("Who holds the MLB career home run record?" + BARE, "Barry Bonds", "short"),
    ("How many innings are in a standard MLB game?\nA. 7\nB. 8\nC. 9\nD. 10", "C", "mc"),
    ("What is it called when a batter hits and reaches all four bases in one play?\nA. Triple\nB. Home run\nC. Grand slam\nD. Cycle", "B", "mc"),
    ("Which league uses the designated hitter in all games as of 2022?\nA. National League\nB. American League\nC. Both\nD. Neither", "C", "mc"),
    ("Who is the all-time MLB hits leader?" + BARE, "Pete Rose", "short"),
    ("How many outs end a half-inning?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
]

OBP_DEF = "OBP = (H + BB + HBP) / (AB + BB + HBP + SF)."
SLG_DEF = "SLG = (singles + 2*2B + 3*3B + 4*HR) / AB, where singles = H - 2B - 3B - HR."
AVG_DEF = "AVG = H / AB."

STATS_ITEMS = [
    (f"Using {OBP_DEF} Career totals: AB=1000, H=300, BB=100, HBP=5, SF=5. What is OBP?{NUM}", "0.365", "stats"),
    (f"Using {OBP_DEF} Career totals: AB=800, H=200, BB=80, HBP=0, SF=0. What is OBP?{NUM}", "0.318", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=500, H=150, 2B=30, 3B=5, HR=20. What is SLG?{NUM}", "0.500", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=400, H=100, 2B=20, 3B=2, HR=10. What is SLG?{NUM}", "0.385", "stats"),
    (f"Using {AVG_DEF} Career totals: AB=600, H=180. What is AVG?{NUM}", "0.300", "stats"),
    (f"Using {OBP_DEF} Career totals: AB=900, H=270, BB=90, HBP=5, SF=5. What is OBP?{NUM}", "0.367", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=700, H=210, 2B=40, 3B=4, HR=30. What is SLG?{NUM}", "0.497", "stats"),
    (f"Using {AVG_DEF} Career totals: AB=300, H=90. What is AVG?{NUM}", "0.300", "stats"),
    (f"Using {OBP_DEF} Career totals: AB=1000, H=250, BB=150, HBP=10, SF=10. What is OBP?{NUM}", "0.350", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=550, H=140, 2B=25, 3B=3, HR=15. What is SLG?{NUM}", "0.393", "stats"),
]

def score_short_loose(pred: str, gold: str) -> bool:
    """Diagnostic only: gold appears as a substring after normalization."""
    from scorer import normalize
    return normalize(gold) in normalize(pred)

print("Loading tokenizer + model (fp16 on T4)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float16, device_map="auto",
)
model.eval()
print(f"Loaded. GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def generate(prompt, max_new_tokens=96):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def run(items, label):
    correct_strict, correct_loose, results = 0, 0, []
    t0 = time.time()
    for q, gold, qtype in items:
        pred = generate(q)
        ok_strict = score(pred, gold, qtype)
        ok_loose  = ok_strict or (qtype != "mc" and score_short_loose(pred, gold))
        correct_strict += ok_strict
        correct_loose  += ok_loose
        results.append((gold, pred, ok_strict, ok_loose, qtype))
    dt = time.time() - t0
    n = len(items)
    print(f"\n=== {label}: strict {correct_strict}/{n} = {correct_strict/n:.0%}"
          f" | loose {correct_loose}/{n} = {correct_loose/n:.0%}  ({dt:.1f}s) ===")
    for gold, pred, ok_s, ok_l, qtype in results:
        mark = "✓" if ok_s else ("~" if ok_l else "✗")   # ✓ strict, ~ loose-only, ✗ miss
        print(f"  {mark} [{qtype:5}] gold={gold!r:>14}  pred={pred[:70]!r}")
    return correct_strict / n, correct_loose / n

print("\n" + "="*60)
know_strict, know_loose = run(KNOWLEDGE_ITEMS, "KNOWLEDGE")
stats_strict, stats_loose = run(STATS_ITEMS, "STATS")

print("\n" + "="*60)
print(f"Knowledge: strict {know_strict:.0%} | loose {know_loose:.0%}")
print(f"Stats:     strict {stats_strict:.0%} | loose {stats_loose:.0%}")

print("\nInterpretation:")
if know_loose - know_strict >= 0.10:
    print(f"  Short items lift {know_loose-know_strict:.0%} with loose matching ->")
    print("  formatting was the miss. Use bare-answer prompts + strict scoring in the real set.")
else:
    print("  Small strict-vs-loose gap -> the short misses are knowledge, not format.")
    print("  Consider making the knowledge slice MC-dominant.")

if stats_strict >= 0.10:
    print(f"  Stats off the floor ({stats_strict:.0%}) with defined formulas ->")
    print("  keep a small stats control (~10-15% of test).")
else:
    print(f"  Stats still at {stats_strict:.0%} even with formulas defined ->")
    print("  the 1B model can't compute these. Keep 5-10 stats items purely as the")
    print("  caveat-demonstrator; do not make stats a headline metric.")

print(f"\nGPU peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Loading tokenizer + model (fp16 on T4)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded. GPU mem after load: 6.06 GB


=== KNOWLEDGE: strict 6/10 = 60% | loose 6/10 = 60%  (24.0s) ===
  ✓ [mc   ] gold=           'B'  pred="The correct answer is **B. 9**\n\nHere's why:\n\n*   **Starting Field Posi"
  ✓ [mc   ] gold=           'B'  pred='The correct answer is **B. 3**\n\nA strikeout is when a pitcher throws a'
  ✓ [mc   ] gold=           'B'  pred='The correct answer is **B. 4**\n\nA walk is typically made up of four ba'
  ✓ [mc   ] gold=           'B'  pred='The correct answer is **B. Chicago Cubs**.\n\nThe Chicago Cubs won the 2'
  ✗ [short] gold= 'Barry Bonds'  pred='Babe Ruth'
  ✗ [mc   ] gold=           'C'  pred="The correct answer is **B. 8**\n\nHere's how innings are calculated in a"
  ✗ [mc   ] gold=           'B'  pred="The correct answer is **D. Cycle**.\n\nHere's why:\n\n*   **Cycle:** This "
  ✓ [mc   ] gold=           'C'  pred='The correct answer is **C. Both**\n\nAs of 2022, the MLB utilized the de'
  ✗ [short] gold=   'Pete Rose'  pred='Babe Ruth'
  